# Iodine: a Morse potential and anharmonic dynamics
Scan I2 with **MACE-OMOL**, fit a Morse well, and compare ASE NVE dynamics on
the fitted Morse potential, with an optional direct MLIP comparison. Iodine is supported by this checkpoint. We use OMOL here because
OFF23-small produced irregular diatomic scans despite supporting the elements.

Run in the MACE kernel. The fit describes this model over the sampled range;
it is not an experimental potential or a validated dissociation energy.
The asymptote and dissociation physics are not established by this finite scan.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.optimize import curve_fit
from ase import Atoms, units
from ase.io import read
from ase.md.verlet import VelocityVerlet
from ase.calculators.morse import MorsePotential
from mace.calculators import MACECalculator
sys.path.insert(0, str(Path.cwd().parent.parent.parent))
from demo_tools.helpers import start_exercise
from demo_tools.view_mols import view
_, OUTPUT = start_exercise()
torch.set_num_threads(int(os.environ.get('SLURM_CPUS_PER_TASK', '4')))
calc = MACECalculator(model_paths='../../models/MACE-omol-0-extra-large-1024.model',
                      device='cpu', default_dtype='float64', head='omol')

## 1. Scan and fit
Change only the bond distance. Fit E(r) = offset + D[1 - exp(-a(r - re))]^2.
D is in eV, a in inverse angstroms, and re in angstroms. Plot the residuals as well:
a fitted curve can conceal features of the original model.

In [ ]:
atoms = Atoms('I2', positions=[[0, 0, 0], [0, 0, 2.7]])
atoms.info.update(total_charge=0, total_spin=0)
atoms.calc = calc
r = np.linspace(2.1, 4.5, 61)
energy = []
for distance in r:
    atoms.positions[1, 2] = distance
    energy.append(atoms.get_potential_energy())
energy = np.array(energy) - np.min(energy)
def morse(r, D, a, re, offset):
    return offset + D*(1 - np.exp(-a*(r - re)))**2
params, _ = curve_fit(morse, r, energy, p0=[2, 1.5, r[energy.argmin()], 0],
                      bounds=([0.01, 0.05, 2.1, -10], [20, 10, 4.5, 10]))
D, a, re, offset = params
print(f'D = {D:.3f} eV; a = {a:.3f} /angstrom; re = {re:.3f} angstrom')
print(f'Fit RMS error = {np.sqrt(np.mean((morse(r, *params)-energy)**2)):.4f} eV')
fig, ax = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
ax[0].plot(r, energy, 'o', ms=3, label='OMOL')
ax[0].plot(r, morse(r, *params), label='Morse fit')
ax[0].plot(r, offset + D*a*a*(r-re)**2, '--', label='Harmonic near minimum')
ax[0].set(ylabel='Relative energy (eV)', ylim=(-0.1, energy.max()*1.1)); ax[0].legend()
ax[1].plot(r, 1000*(energy-morse(r, *params)))
ax[1].set(xlabel='Bond distance (angstrom)', ylabel='Residual (meV)')
plt.tight_layout(); plt.show()
np.savetxt(OUTPUT/'potential.csv', np.column_stack((r, energy, morse(r, *params))),
           delimiter=',', header='r_A,omol_relative_eV,morse_relative_eV', comments='')

## 2. Stretch and release
Start both atoms at rest using the bond extensions in the loop below.
Run without a thermostat for `STEPS * DT_FS / 1000` ps.
This isolates bond vibration; no thermal velocities or temperature target are needed.
The analytic Morse dynamics are fast. Set `COMPARE_MLIP = True` to also run the
more expensive OMOL trajectories.
ASE's Morse calculator uses rho0 = a*re and sets the well minimum to -D.
This constant energy offset does not change the forces. Its cutoff starts beyond
our scan and the bound trajectories.

In [ ]:
morse_calc = MorsePotential(epsilon=D, rho0=a*re, r0=re, rcut1=3, rcut2=4)
DT_FS = 0.5
STEPS = 1000
SAVE_EVERY = 5
COMPARE_MLIP = True
calculators = [('Morse', morse_calc)]
if COMPARE_MLIP:
    calculators.append(('OMOL', calc))
runs = {}
for label, calculator in calculators:
    for stretch in [0.05, 1.0]:
        distance = re + stretch
        atoms = Atoms('I2', positions=[[0, 0, -distance/2], [0, 0, distance/2]])
        atoms.info.update(total_charge=0, total_spin=0)
        atoms.calc = calculator
        atoms.set_momenta(np.zeros((2, 3)))
        path = OUTPUT/f'{label}_{stretch:.2f}.traj'
        dyn = VelocityVerlet(atoms, DT_FS*units.fs, trajectory=str(path), loginterval=SAVE_EVERY)
        dyn.run(STEPS)
        runs[label, stretch] = read(path, index=':')

## 3. Compare the motion
Larger-amplitude Morse motion should have a longer period and spend more time on
the stretched side. Compare OMOL with its fit rather than assuming they agree.
Energy changes below are relative to each trajectory's own initial energy.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
for (label, stretch), frames in runs.items():
    time_fs = np.arange(len(frames))*SAVE_EVERY*DT_FS
    distance = np.array([frame.get_distance(0, 1) for frame in frames])
    total = np.array([frame.get_total_energy() for frame in frames])
    ax[0].plot(time_fs, distance, '--' if label == 'Morse' else '-', label=f'{label}, +{stretch:.2f} A')
    ax[1].plot(time_fs, 1000*(total-total[0]))
    print(label, stretch, 'bond range:', distance.min(), distance.max(), 'angstrom')
ax[0].set_ylabel('Bond distance (angstrom)'); ax[0].legend()
ax[1].set(xlabel='Time (fs)', ylabel='Total energy change (meV)')
plt.tight_layout(); plt.show()
view(next(reversed(runs.values())), animate=True)

## Try and explain
- Why is the harmonic approximation best near the minimum?
- Does the oscillation period increase with amplitude? Does OMOL follow the Morse prediction?
- Why are kinetic and potential energies changing while their sum stays approximately fixed?
- Would a good fit over this interval establish the true dissociation energy?